# Code mẫu — Pipeline và data leakage

Median để điền ô thiếu, thang đo của scaler, danh sách giá trị của encoder, tất cả đều là những thứ được học từ dữ liệu. Vì vậy chúng phải được học ở cùng phía với model, tức là trên train, rồi giữ nguyên khi biến đổi mẫu mới. Pipeline là công cụ giúp làm việc đó một cách tự động.

## Ví dụ 1: Validation lén làm đổi median và cả thước đo

Train có hai giá trị số đăng ký là 10 và 20, cùng một ô thiếu. Validation có một buổi với 100 người đăng ký. Ta so cách fit chỉ trên train với cách gộp cả hai bảng lại. Sau đó xem vì sao fit scaler riêng trên validation làm mất hẳn ý nghĩa của con số 100.

In [ ]:
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

train = np.array([[10.0], [20.0], [np.nan]])
valid = np.array([[100.0]])
correct = SimpleImputer(strategy="median").fit(train)
leaked = SimpleImputer(strategy="median").fit(np.vstack([train, valid]))
print("Median chỉ train:", correct.statistics_.tolist())
print("Median có nhìn validation, cách sai:", leaked.statistics_.tolist())
print("Train điền đúng:", correct.transform(train).ravel().tolist())

scaler = StandardScaler().fit([[10.0], [20.0]])
print("100 dùng thang train:", scaler.transform(valid).ravel().tolist())
print("100 fit thang riêng, cách sai:", StandardScaler().fit_transform(valid).ravel().tolist())

Median chỉ train: [15.0]
Median có nhìn validation, cách sai: [20.0]
Train điền đúng: [10.0, 20.0, 15.0]
100 dùng thang train: [17.0]
100 fit thang riêng, cách sai: [0.0]


Median đúng là 15. Gộp thêm validation làm nó thành 20, dù ta chưa hề đụng tới nhãn của validation. Rò rỉ không cần đến nhãn, chỉ cần thống kê của tập đánh giá lọt vào bước học. Train có mean 15 và độ lệch chuẩn 5, nên 100 được đổi thành `(100 − 15) / 5 = 17`, tức một buổi đông bất thường. Fit riêng trên một mẫu duy nhất lại biến 100 thành 0, như thể đó là một buổi bình thường, vì thước đo đã khác hẳn thước đo model được học.

## Ví dụ 2: Bảng có cả số, chữ, và một danh mục chưa từng gặp

Sáu buổi train giả lập có số người đến bằng hai lần số đăng ký, cộng thêm 5 nếu là buổi chiều. Một ô số đăng ký bị thiếu, nhưng target vẫn được ghi lại. Pipeline điền median, chuẩn hóa cột số, one-hot cột thời điểm, rồi fit LinearRegression.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression

train = pd.DataFrame({
    "registered": [10.0, 20.0, np.nan, 10.0, 20.0, 30.0],
    "slot": ["sang", "sang", "sang", "chieu", "chieu", "chieu"],
})
y = np.array([20, 40, 40, 25, 45, 65])
numeric = Pipeline([
    ("fill", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
])
preprocess = ColumnTransformer([
    ("num", numeric, ["registered"]),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), ["slot"]),
])
model = Pipeline([("prep", preprocess), ("regression", LinearRegression())])
model.fit(train, y)
fitted_prep = model.named_steps["prep"]
median = fitted_prep.named_transformers_["num"].named_steps["fill"].statistics_[0]
print("Median train đã học:", float(median))
print("Tên feature sau biến đổi:", fitted_prep.get_feature_names_out().tolist())
assert median == train["registered"].median()

Median train đã học: 20.0
Tên feature sau biến đổi: ['num__registered', 'cat__slot_chieu', 'cat__slot_sang']


Median 20 được học từ năm ô số có giá trị trong train. Một cột số cùng hai cột one-hot cho thời điểm tạo thành ba feature. ID và target không đi qua bộ biến đổi nào. In tên feature ra như vậy là cách nhanh nhất để chắc pipeline đang xử lý đúng các cột mình nghĩ.

Bây giờ, giữ nguyên model để dự đoán một buổi sáng và một buổi tối, cùng có 25 người đăng ký. Buổi tối chưa từng xuất hiện trong train. Encoder vẫn cho chạy tiếp, nhưng chạy được không có nghĩa là model đã học được buổi tối khác buổi sáng ra sao.

In [ ]:
new = pd.DataFrame({"registered": [25.0, 25.0], "slot": ["sang", "toi"]})
transformed = fitted_prep.transform(new)
print("Shape mới:", transformed.shape)
print("One-hot buổi tối:", transformed[1, 1:].tolist())
print("Dự đoán sáng/tối:", np.round(model.predict(new), 2).tolist())

Shape mới: (2, 3)
One-hot buổi tối: [0.0, 0.0]
Dự đoán sáng/tối: [50.0, 52.5]


Buổi sáng có cơ chế đã được quan sát, nên dự đoán 50 người là hợp lý. Buổi tối được mã hóa thành `[0, 0]`, nên con số 52.5 chỉ là kết quả của việc model kéo dài các hệ số đã học, chứ không phải một ước lượng về buổi tối đã được kiểm chứng. Pipeline cũng không tự phát hiện được một cột chỉ đo được sau sự kiện. Feature nào hợp lệ vẫn phải do người làm bài tự xét, dựa trên việc thông tin nào có sẵn lúc dự đoán.

## Ví dụ 3: Kiểm median được học trong từng fold

Mười hai buổi xếp theo thời gian có số đăng ký tăng dần, với hai ô bị thiếu. Nếu fit imputer một lần trên cả bảng rồi mới làm cross-validation, các fold đầu sẽ được nhìn thông tin từ những buổi về sau. Cách đúng là đưa cả Pipeline vào `cross_validate`, rồi đọc lại imputer đã được fit trong từng fold để kiểm.

In [ ]:
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.model_selection import TimeSeriesSplit, cross_validate

X = np.arange(10, 130, 10, dtype=float).reshape(-1, 1)
y = 2 * X[:, 0] + 5
X[[2, 8], 0] = np.nan
pipeline = Pipeline([
    ("fill", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
    ("regression", Ridge(alpha=1.0)),
])
folds = list(TimeSeriesSplit(n_splits=3, test_size=2).split(X))
result = cross_validate(
    pipeline, X, y, cv=folds, scoring="neg_mean_absolute_error",
    return_estimator=True, n_jobs=1,
)
for fold, ((train_ids, valid_ids), estimator) in enumerate(zip(folds, result["estimator"]), start=1):
    learned = float(estimator.named_steps["fill"].statistics_[0])
    expected = float(np.nanmedian(X[train_ids, 0]))
    assert learned == expected
    print(f"Fold {fold}: train đến hàng {int(train_ids[-1])}, median={learned:.1f}, kiểm train={expected:.1f}")
print("Median toàn bảng, không dùng cho fold:", float(np.nanmedian(X[:, 0])))

Fold 1: train đến hàng 5, median=40.0, kiểm train=40.0
Fold 2: train đến hàng 7, median=50.0, kiểm train=50.0
Fold 3: train đến hàng 9, median=55.0, kiểm train=55.0
Median toàn bảng, không dùng cho fold: 65.0


Median thay đổi theo phần quá khứ của từng fold: 40, rồi 50, rồi 55. Median của cả bảng là 65, tức đã nhìn tới cả những buổi trong tương lai. Tham số `return_estimator=True` cho phép ta mở từng model đã fit ra để kiểm trạng thái nó thật sự học được. Chỉ in một điểm CV đẹp thì chưa chứng minh được imputer đã tôn trọng ranh giới giữa train và validation.